# RL Experiment 02: DQN Baseline

**Self-contained experiment notebook using DRY architecture.**

## Architecture (DRY Principle)

| Location | What | Example |
|----------|------|---------|
| `schedule_engine/notebooks/` | Reusable functions | `load_context()`, `create_env()`, `train_agent()` |
| `src/schedule_engine/rl/` | Production RL components | `ScheduleEnv`, PPO/DQN agents |
| **This notebook** | Experiment-specific config | `TIMESTEPS`, `SEED`, execution |

## Experiment Overview
- **Agent**: DQN (Deep Q-Network)
- **Goal**: Compare DQN performance against PPO baseline
- **Metrics**: Best fitness, convergence generation, training time

## 1. Imports (from `schedule_engine/notebooks/`)

In [ ]:
from __future__ import annotations
from datetime import datetime
from pathlib import Path

# DRY IMPORTS FROM schedule_engine/notebooks/
from schedule_engine.notebooks import (
    build_notebook_config,
    create_env,
    evaluate_agent,
    load_context,
    set_global_seed,
    train_agent,
)

print(" All imports from schedule_engine/notebooks/ successful!")

## 2. Configuration (Inline - Experiment-Specific)

In [ ]:
# ============================================================================
# RL EXPERIMENT 02 CONFIGURATION - Modify these as needed
# ============================================================================

SEED = 42
POP_SIZE = 20
MAX_GENERATIONS = 50
MAX_STEPS = 20
TIMESTEPS = 5000  # Training timesteps for DQN

# Paths - Organized by experiment with timestamp
TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
DATA_DIR = Path("../data")
OUTPUT_DIR = Path(f"../output/notebooks/rl_02_dqn_baseline_{TIMESTAMP}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f" Config: pop={POP_SIZE}, ngen={MAX_GENERATIONS}, steps={MAX_STEPS}, timesteps={TIMESTEPS}")
print(f" Output: {OUTPUT_DIR}")

## 3. Load Data & Create Environment

In [ ]:
# Set reproducibility
set_global_seed(SEED)

# Build config and load scheduling context
config = build_notebook_config(seed=SEED, overrides={"pop_size": POP_SIZE})
_, context = load_context(DATA_DIR, config)

# Create RL environment with production ScheduleEnv
env = create_env(
    context=context,
    pop_size=POP_SIZE,
    max_generations=MAX_GENERATIONS,
    max_steps=MAX_STEPS,
)

print(f" Environment created: obs_space={env.observation_space.shape}, action_space={env.action_space.n}")

## 4. Train DQN Agent

In [ ]:
# Train DQN agent on ScheduleEnv
agent, train_time = train_agent(
    agent_type="dqn",
    env=env,
    timesteps=TIMESTEPS,
    seed=SEED,
)

print(f" DQN agent trained in {train_time:.2f}s")

## 5. Evaluate Agent

In [ ]:
# Evaluate trained agent
result = evaluate_agent(agent, env, max_generations=MAX_GENERATIONS)

print(f"\n{'='*50}")
print(f"RL EXPERIMENT 02: DQN BASELINE RESULTS")
print(f"{'='*50}")
print(f"Training time: {train_time:.2f}s")
print(f"Best fitness:  {result.best_fitness}")
print(f"Convergence:   Generation {result.convergence_gen}/{MAX_GENERATIONS}")
print(f"{'='*50}")

## 6. Save Results

In [ ]:
import json

# Save experiment results
results_data = {
    "experiment": "rl_02_dqn_baseline",
    "timestamp": TIMESTAMP,
    "config": {
        "seed": SEED,
        "pop_size": POP_SIZE,
        "max_generations": MAX_GENERATIONS,
        "max_steps": MAX_STEPS,
        "timesteps": TIMESTEPS,
        "agent_type": "dqn",
    },
    "results": {
        "train_time_seconds": train_time,
        "best_fitness": result.best_fitness,
        "convergence_gen": result.convergence_gen,
    },
}

results_path = OUTPUT_DIR / "results.json"
with open(results_path, "w") as f:
    json.dump(results_data, f, indent=2)

print(f" Results saved to: {results_path}")